# Notebook 1 — Raccolta Dati
**Tesi Magistrale**: Referendum Costituzionale Giustizia 2026 — Separazione delle Carriere

Questo notebook contiene **tutta la pipeline di raccolta dati** per la tesi.

| Sezione | Cosa fa | Output |
|---------|---------|--------|
| **§1** Schema annotazione | Definisce le etichette STANCE (POS/NEG/NEU) | — |
| **§2** YouTube Referendum | Scraping incrementale commenti su video pertinenti | `referendum_giustizia_full_debate.csv` |
| **§3** YouTube Governo | Scraping con tracking quota API (10k/giorno) | `governo_meloni_consenso.csv` |
| **§4** Stampa | Caricamento dataset articoli (raccolto esternamente) | `dataset_articoli_referendum.csv` |
| **§5** Contesto padre-figlio | Aggiunge `Testo_Padre` alle risposte per il modello BERT | `referendum_con_padre.csv` |

**API**: YouTube Data API v3 — quota giornaliera 10.000 unità (reset 09:00 ora italiana)
**Modalità**: incrementale — ad ogni esecuzione scarica solo i commenti **nuovi** (delta)
**Deduplicazione**: chiave `(video_id, autore, data)` per evitare doppioni tra esecuzioni

## §1 — Schema di Annotazione: STANCE

La variabile **STANCE** classifica la posizione del commentatore rispetto al referendum sulla separazione delle carriere in magistratura.

### Etichette aggregate (usate per il training BERT)

| Codice | Etichetta | Descrizione |
|--------|-----------|-------------|
| **POS** | Favorevole | Supporto esplicito o implicito alla riforma |
| **NEG** | Contrario | Opposizione (timore indebolimento azione penale, influenza politica) |
| **NEU** | Neutro/Disilluso | Nessuna posizione chiara, sfiducia istituzionale generica, off-topic |

### Sub-categorie originali (granularità di annotazione, poi aggregate in POS/NEG/NEU)

| Codice | Etichetta | Descrizione |
|--------|-----------|-------------|
| **S1** | Favorevole | Argomenti pro-riforma: efficienza, indipendenza, accountability |
| **S2** | Contrario | Argomenti anti-riforma: timori azione penale, influenza politica |
| **S3** | Disilluso | Sfiducia generalizzata verso istituzioni (→ NEU) |
| **S4** | Neutro | Osservazioni descrittive o fattuali (→ NEU) |
| **S5** | Off-topic | Estraneo al tema (→ NEU, escluso dal training in alcuni esperimenti) |

> **Nota**: nella prima fase di progettazione era prevista anche una variabile `FRAME` (cornice argomentativa: Casta, Democrazia, Complotto, Tecnico, Partitico, Emotivo, Ironico, Meta) ma è stata **abbandonata**. La tesi finale usa **solo la variabile STANCE**.

## §2 — Raccolta YouTube: Referendum Giustizia

Pipeline a 3 step:
1. **Stato persistente** — carica il CSV esistente per abilitare modalità delta (scarica solo commenti dopo l'ultimo salvato).
2. **Raccolta video** — query mirate sul referendum, filtro per pertinenza (lessico tematico) e rumore (noise keywords).
3. **Estrazione commenti** — per ogni video chiama `commentThreads.list`; per video già noti usa `publishedAfter` per scaricare solo il delta.

**Query** coprono il referendum sulla separazione delle carriere, includendo anche varianti per Shorts (formato breve).
**Filtro pertinenza**: il titolo o la descrizione devono contenere almeno una keyword del lessico (referendum, giustizia, magistrat, csm, ...) e nessuna noise (fiction, calcio, makeup, ...).

### §2.1 — Setup, configurazione, filtri

In [ ]:
import pandas as pd
from googleapiclient.discovery import build
from kaggle_secrets import UserSecretsClient
import time, re, os
from datetime import datetime, timezone

# ── CONFIGURAZIONE ────────────────────────────────────────────────────────────
# La API key è salvata come secret di Kaggle per non esporla nel notebook
user_secrets = UserSecretsClient()
API_KEY  = user_secrets.get_secret("YOUTUBE_API")
youtube  = build("youtube", "v3", developerKey=API_KEY)

# Path: input read-only (Kaggle dataset versione precedente) e working (dove salviamo l'aggiornato)
INPUT_FILE  = "/kaggle/input/datasets/francescochiarolanza/commenti-youtube-referendum-giustizia/Dataset Tesi Aggiornato.csv"
OUTPUT_FILE = "/kaggle/working/Dataset Tesi Aggiornato.csv"

# Query: combinazione di parole chiave specifiche del referendum + varianti shorts
# (la versione "shorts" intercetta i video brevi che hanno linguaggio diverso)
QUERIES = [
    "referendum giustizia 2026",
    "separazione carriere magistrati",
    "riforma giustizia",
    "commenti referendum giustizia",
    "referendum giustizia shorts",
    "separazione carriere shorts",
    "riforma giustizia 2026 shorts",
]

# Lessico di pertinenza: il video è incluso se titolo/descrizione contengono almeno una di queste
RELEVANCE_KEYWORDS = [
    "referendum", "giustizia", "magistrat", "separazione carriere",
    "riforma", "csm", "toghe", "procura", "giudic", "pm ",
    "corte", "costituzion", "voto", "quesito",
]

# Noise keywords: il video è escluso se contiene una di queste (anche se ha pertinenza)
# Serve a filtrare contenuti omonimi o totalmente off-topic che usano gli stessi termini
NOISE_KEYWORDS = [
    "fiction", "film", "serie tv", "musica", "canzone", "ricetta",
    "tutorial trucco", "makeup", "calcio", "sport",
]

# Limiti per esecuzione (per non sforare quota API in una sola run)
MAX_PAGES_PER_QUERY = 10   # pagine search.list per ogni query (50 video/pagina)
MAX_COMMENT_PAGES   = 20   # pagine commentThreads.list per ogni video (100 commenti/pagina)


def is_relevant(title, description=""):
    """True se il video parla di referendum giustizia. Controlla keyword di pertinenza E assenza di noise."""
    text = (title + " " + description).lower()
    if any(noise in text for noise in NOISE_KEYWORDS):
        return False
    return any(kw in text for kw in RELEVANCE_KEYWORDS)


def is_short(duration_iso):
    """True se la durata ISO 8601 (es. 'PT45S' o 'PT1M30S') è ≤ 60 secondi → YouTube Short."""
    m = re.match(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", duration_iso)
    if not m:
        return False
    h, mn, s = int(m.group(1) or 0), int(m.group(2) or 0), int(m.group(3) or 0)
    return h * 3600 + mn * 60 + s <= 60

### §2.2 — Stato persistente e modalità delta

`load_existing_state` carica il CSV esistente per:
- ricavare l'elenco di **video già visti** (`known_ids`)
- la **data dell'ultimo commento salvato** per ogni video (`last_dates`) → usata in `commentThreads.list(publishedAfter=...)` per scaricare solo il delta
- una **chiave di deduplicazione** `(video_id, autore, data)` per evitare doppioni residui

Priorità: `/kaggle/working` (versione aggiornata di questa run) > `/kaggle/input` (read-only).

In [ ]:
def load_existing_state(input_path, output_path):
    """Carica stato del CSV esistente per la modalità delta."""
    filepath = output_path if os.path.exists(output_path) else (
               input_path  if os.path.exists(input_path)  else None)
    if filepath is None:
        print("Partenza da zero (nessun CSV esistente).")
        return pd.DataFrame(), set(), {}, set()

    print(f"Carico stato esistente da: {filepath}")
    df = pd.read_csv(filepath, encoding="utf-8-sig", dtype=str, sep=";",
                     on_bad_lines="skip", engine="python")
    df["Data_Commento"] = pd.to_datetime(df["Data_Commento"], utc=True, errors="coerce")

    known_ids  = set(df["Video_ID"].dropna().unique())
    # Per ogni video, l'ultima data salvata → la API filtrerà server-side con publishedAfter
    last_dates = (df.groupby("Video_ID")["Data_Commento"].max().dropna()
                    .dt.strftime("%Y-%m-%dT%H:%M:%SZ").to_dict())
    # Set di chiavi (video, autore, data) per scartare duplicati residui
    known_keys = set(zip(df["Video_ID"].fillna(""),
                         df["Autore"].fillna(""),
                         df["Data_Commento"].astype(str)))

    print(f"  → {len(df):,} righe, {len(known_ids)} video già presenti")
    return df, known_ids, last_dates, known_keys

### §2.3 — Raccolta video ID (search.list)

Per ogni query: itera le pagine, applica `is_relevant`, accumula i video unici nel `video_map`.

Il costo di `search.list` è **100 unità di quota** per chiamata: con 7 query × 10 pagine = 7000 unità potenziali. Lasciamo margine per `commentThreads.list` (1 unità/chiamata).

In [ ]:
def collect_all_video_ids(queries):
    """Per ogni query: itera le pagine search.list e raccoglie i video pertinenti."""
    video_map, seen = {}, set()
    for q in queries:
        print(f"Query: '{q}'")
        page_token, page_count = None, 0
        while page_count < MAX_PAGES_PER_QUERY:
            try:
                resp = youtube.search().list(
                    q=q, part="id,snippet", maxResults=50,
                    type="video", order="relevance", pageToken=page_token
                ).execute()
            except Exception as e:
                print(f"  Errore: {e}")
                break

            for item in resp.get("items", []):
                v_id  = item["id"]["videoId"]
                if v_id in seen:
                    continue
                title = item["snippet"]["title"]
                desc  = item["snippet"].get("description", "")
                if not is_relevant(title, desc):
                    continue
                seen.add(v_id)
                video_map[v_id] = {
                    "Titolo_Video": title,
                    "Canale":       item["snippet"]["channelTitle"],
                    "Link":         f"https://www.youtube.com/watch?v={v_id}",
                    "Data_Video":   item["snippet"]["publishedAt"],
                    "Is_Short":     None,   # arricchito dopo con videos.list
                }

            page_count += 1
            page_token  = resp.get("nextPageToken")
            print(f"  pag. {page_count}: {len(video_map)} video totali pertinenti")
            if not page_token:
                break
            time.sleep(0.3)   # piccola pausa per evitare rate limit
    return video_map

### §2.4 — Arricchimento con durata (videos.list → Is_Short)

`search.list` non restituisce la durata. Serve una chiamata separata a `videos.list(part='contentDetails')` per ottenere `duration` (formato ISO 8601, es. `PT2M15S`).

Batch da 50 video/chiamata → minimizza il numero di richieste.

In [ ]:
def enrich_with_duration(video_map):
    """Aggiunge Is_Short e Durata_ISO al video_map."""
    ids = list(video_map.keys())
    for i in range(0, len(ids), 50):
        batch = ids[i:i+50]
        try:
            resp = youtube.videos().list(part="contentDetails", id=",".join(batch)).execute()
            for item in resp.get("items", []):
                dur = item["contentDetails"]["duration"]
                video_map[item["id"]]["Is_Short"]   = is_short(dur)
                video_map[item["id"]]["Durata_ISO"] = dur
        except Exception as e:
            print(f"  Errore arricchimento durata: {e}")
        time.sleep(0.2)
    shorts = sum(1 for v in video_map.values() if v.get("Is_Short"))
    print(f"  Shorts identificati: {shorts}/{len(ids)}")
    return video_map

### §2.5 — Estrazione commenti (modalità delta)

Per ogni video:
- **NUOVO** (non in `known_ids`): scarica tutti i commenti.
- **DELTA** (già noto): passa `publishedAfter=last_date` → la API restituisce solo i commenti dopo l'ultimo salvato.

Estrae sia commenti **principali** che **risposte** (queste ultime con `Padre_ID` per l'associazione successiva).
Stop anticipato: se in una pagina non ci sono nuovi commenti per un video noto, smette (riduce chiamate inutili).

In [ ]:
def extract_comments(video_map, known_ids, last_dates, known_keys):
    """Estrae commenti per tutti i video. Modalità DELTA per video già noti."""
    new_rows = []
    for idx, (v_id, meta) in enumerate(video_map.items(), 1):
        is_known  = v_id in known_ids
        last_date = last_dates.get(v_id)
        label     = "Short" if meta.get("Is_Short") else "Video"
        mode      = "DELTA" if is_known else "NUOVO"
        print(f"[{idx}/{len(video_map)}] {label} [{mode}] → {meta['Titolo_Video'][:50]}...")

        try:
            page_token, page_count, stop_early = None, 0, False
            while page_count < MAX_COMMENT_PAGES and not stop_early:
                kwargs = dict(
                    part="snippet,replies", videoId=v_id, maxResults=100,
                    textFormat="plainText", order="time", pageToken=page_token
                )
                # Se il video è noto, filtra lato server con publishedAfter (delta)
                if is_known and last_date:
                    kwargs["publishedAfter"] = last_date

                resp = youtube.commentThreads().list(**kwargs).execute()
                page_new = 0

                for item in resp["items"]:
                    # Commento principale (top-level)
                    top  = item["snippet"]["topLevelComment"]["snippet"]
                    ckey = (v_id, top["authorDisplayName"],
                            str(pd.to_datetime(top["publishedAt"], utc=True)))
                    if ckey not in known_keys:
                        known_keys.add(ckey)
                        new_rows.append({
                            **meta, "Video_ID": v_id,
                            "Tipo_Commento": "Principale", "Padre_ID": None,
                            "Autore":        top["authorDisplayName"],
                            "Testo":         top["textDisplay"],
                            "Likes":         top["likeCount"],
                            "Data_Commento": top["publishedAt"],
                        })
                        page_new += 1

                    # Risposte al commento principale (se presenti)
                    if "replies" in item:
                        for reply in item["replies"]["comments"]:
                            r    = reply["snippet"]
                            rkey = (v_id, r["authorDisplayName"],
                                    str(pd.to_datetime(r["publishedAt"], utc=True)))
                            if rkey not in known_keys:
                                known_keys.add(rkey)
                                new_rows.append({
                                    **meta, "Video_ID": v_id,
                                    "Tipo_Commento": "Risposta",
                                    "Padre_ID":      item["id"],
                                    "Autore":        r["authorDisplayName"],
                                    "Testo":         r["textDisplay"],
                                    "Likes":         r["likeCount"],
                                    "Data_Commento": r["publishedAt"],
                                })
                                page_new += 1

                # Stop anticipato: video noto e nessun nuovo commento → non serve continuare
                if is_known and page_new == 0:
                    stop_early = True

                page_count += 1
                page_token  = resp.get("nextPageToken")
                if not page_token:
                    break
                time.sleep(0.2)

        except Exception as e:
            print(f"  Commenti non disponibili: {e}")

    print(f"\nNuovi commenti raccolti: {len(new_rows):,}")
    return pd.DataFrame(new_rows)

### §2.6 — Esecuzione pipeline referendum

Sequenza:
1. `load_existing_state` → stato delta
2. `collect_all_video_ids(QUERIES)` → mappa video pertinenti
3. `enrich_with_duration(video_map)` → aggiunge Is_Short
4. `extract_comments(...)` → scarica commenti (delta o full)
5. **Merge + dedup finale** → concat con esistente, drop_duplicates su (Video_ID, Autore, Data, Testo)
6. **Save** → CSV in `/kaggle/working/` (con sep=`;` per compatibilità con la pipeline successiva)

In [ ]:
df_existing, known_ids, last_dates, known_keys = load_existing_state(INPUT_FILE, OUTPUT_FILE)
video_map = collect_all_video_ids(QUERIES)
video_map = enrich_with_duration(video_map)

# Stats: quanti video sono nuovi vs già noti
n_new   = sum(1 for v in video_map if v not in known_ids)
n_delta = sum(1 for v in video_map if v     in known_ids)
print(f"\nVideo trovati: {len(video_map)} totali ({n_new} nuovi, {n_delta} delta)")

df_new = extract_comments(video_map, known_ids, last_dates, known_keys)

if df_new.empty:
    print("\nNessun nuovo commento. Dataset già aggiornato.")
    if not os.path.exists(OUTPUT_FILE) and not df_existing.empty:
        df_existing.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
else:
    df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    # Dedup finale di sicurezza (catchall: rimuove eventuali doppioni residui)
    before = len(df_combined)
    df_combined.drop_duplicates(
        subset=["Video_ID", "Autore", "Data_Commento", "Testo"], inplace=True
    )
    if before != len(df_combined):
        print(f"  Rimossi {before - len(df_combined)} duplicati residui.")
    df_combined.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(f"\nDataset aggiornato: {len(df_combined):,} righe ({len(df_new):,} nuove).")
    print(f"File salvato: {OUTPUT_FILE}")

## §3 — Raccolta YouTube: Governo Meloni

Pipeline analoga al referendum, con due differenze:

1. **Quota tracking esplicito** — la quota giornaliera YouTube è 10.000 unità. Qui teniamo conto del consumo (`SEARCH_COST=100`, `COMMENT_COST=1`) e fermiamo le ricerche quando il budget search è esaurito (limite 5000 unità su search per lasciare margine ai commenti).
2. **Filtro temporale `publishedAfter`** — solo video pubblicati dopo il `2025-09-01` (insediamento del governo nel periodo monitorato).

**Query** più ampie del caso referendum: includono nomi di programmi TV (DiMartedì, Otto e Mezzo, Piazzapulita, Porta a Porta...) e nomi di esponenti (Salvini, Tajani, Schlein, Conte, Nordio...) per catturare il dibattito politico.

**Lessico di pertinenza** centrato su: governo/Meloni/maggioranza/manovra/sondaggi e nomi di partiti.

In [ ]:
import pandas as pd
from googleapiclient.discovery import build
from kaggle_secrets import UserSecretsClient
import time, re, os, sys

# ── CONFIGURAZIONE ────────────────────────────────────────────────────────────
user_secrets = UserSecretsClient()
API_KEY = user_secrets.get_secret("YOUTUBE_API")
youtube = build("youtube", "v3", developerKey=API_KEY)

OUTPUT_FILE     = "governo_meloni_consenso.csv"
PUBLISHED_AFTER = "2025-09-01T00:00:00Z"   # solo video post-insediamento

# ── GESTIONE QUOTA API ────────────────────────────────────────────────────────
# La YouTube API ha 10.000 unità/giorno. search costa 100, commentThreads costa 1.
# Per fare 50 query × 3 pagine = 15.000 unità → rischio quota. Limitiamo il
# budget search a 5000 (50 query) per lasciare il resto ai commenti.
QUOTA_LIMIT         = 10000
SEARCH_COST         = 100
COMMENT_COST        = 1
VIDEO_COST          = 1
QUOTA_SEARCH_BUDGET = 5000
quota_used          = 0
quota_exhausted     = False


def track_quota(cost, op=""):
    """Aggiorna il contatore globale e logga ogni 500 unità consumate."""
    global quota_used
    quota_used += cost
    if quota_used % 500 < cost:
        print(f"  [Quota: {quota_used}/{QUOTA_LIMIT}] ({op})")


def quota_remaining():
    return QUOTA_LIMIT - quota_used


def check_quota_error(e):
    """Riconosce l'errore di quota esaurita dalla API."""
    global quota_exhausted
    if "quotaExceeded" in str(e):
        quota_exhausted = True
        return True
    return False


# Verifica preliminare: se la quota è già esaurita all'inizio, abortiamo
print("Verifica quota API...")
try:
    youtube.videos().list(part="id", id="dQw4w9WgXcQ").execute()
    print("  Quota disponibile.")
except Exception as e:
    if "quotaExceeded" in str(e):
        print("QUOTA ESAURITA. Riprova domani alle 09:00 (ora italiana).")
        sys.exit()
    else:
        print(f"  Errore non-quota: {e}")


# ── QUERY ─────────────────────────────────────────────────────────────────────
# Mix di:
#  - Query istituzionali (governo, fiducia, manovra, sondaggi)
#  - Programmi TV politici (DiMartedì, Otto e Mezzo, Piazzapulita, Porta a Porta)
#  - Esponenti politici (Salvini, Tajani, Schlein, Conte, Nordio, Piantedosi)
#  - Varianti shorts
QUERIES = [
    "governo Meloni", "Meloni governo Italia", "fiducia governo Meloni",
    "sondaggi governo Meloni", "crisi governo Italia", "politica italiana",
    "DiMartedi Meloni", "Otto e mezzo governo", "Piazzapulita Meloni",
    "Porta a Porta governo", "Floris governo Meloni", "Gruber Meloni",
    "manovra governo Meloni", "legge di bilancio Italia",
    "autonomia differenziata", "premierato riforma",
    "Meloni economia lavoro", "Meloni immigrazione",
    "Salvini vicepremier", "Tajani forza italia",
    "Nordio giustizia", "Piantedosi sicurezza",
    "Meloni shorts", "governo Italia shorts",
]
MAX_PAGES_PER_QUERY = 3
MAX_COMMENT_PAGES   = 15
MANUAL_VIDEO_IDS    = []   # opzionale: aggiungi qui ID di video specifici che la search potrebbe perdere

RELEVANCE_KEYWORDS = [
    "governo", "meloni", "premier", "presidente del consiglio",
    "maggioranza", "opposizione", "fiducia", "manovra", "cdm",
    "legge di bilancio", "decreto", "fratelli d\'italia", "fdi", "lega", "forza italia",
    "pd", "m5s", "centrodestra", "centrosinistra",
    "salvini", "tajani", "schlein", "conte", "nordio", "piantedosi", "crosetto",
    "politica", "sondaggi", "consenso", "elezioni", "riforma", "economia",
]
NOISE_KEYWORDS = [
    "fiction", "film completo", "serie tv", "musica", "ricetta",
    "tutorial trucco", "makeup", "calcio serie a", "highlights gol",
    "trailer", "gaming", "fortnite", "minecraft", "asmr",
]


def is_relevant(title, description=""):
    text = (title + " " + description).lower()
    return (not any(n in text for n in NOISE_KEYWORDS) and
            any(kw in text for kw in RELEVANCE_KEYWORDS))


def is_short(duration_iso):
    m = re.match(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", duration_iso)
    if not m:
        return False
    h, mn, s = int(m.group(1) or 0), int(m.group(2) or 0), int(m.group(3) or 0)
    return h * 3600 + mn * 60 + s <= 60

### §3.1 — Funzioni di raccolta (con quota tracking)

In [ ]:
def load_existing_state(filepath):
    """Carica stato del CSV (versione semplificata per il dataset governo)."""
    if not os.path.exists(filepath):
        print("Partenza da zero.")
        return pd.DataFrame(), set(), {}, set()

    df = pd.read_csv(filepath, encoding="utf-8-sig", dtype=str)
    df["Data_Commento"] = pd.to_datetime(df["Data_Commento"], utc=True, errors="coerce")
    known_ids  = set(df["Video_ID"].dropna().unique())
    last_dates = (df.groupby("Video_ID")["Data_Commento"].max().dropna()
                    .dt.strftime("%Y-%m-%dT%H:%M:%SZ").to_dict())
    known_keys = set(zip(df["Video_ID"].fillna(""),
                         df["Autore"].fillna(""),
                         df["Data_Commento"].astype(str)))
    print(f"Esistente: {len(df):,} righe, {len(known_ids)} video")
    return df, known_ids, last_dates, known_keys


def collect_video_ids(queries, known_ids):
    """Raccoglie video con quota tracking: si ferma se il budget search è esaurito."""
    global quota_exhausted
    video_map, seen = {}, set(known_ids)
    sq_used = 0
    for q in queries:
        if quota_exhausted or sq_used + SEARCH_COST > QUOTA_SEARCH_BUDGET:
            print("  Budget search esaurito.")
            break
        print(f"'{q}'")
        page_token, page_count = None, 0
        while page_count < MAX_PAGES_PER_QUERY:
            if quota_exhausted:
                break
            try:
                resp = youtube.search().list(
                    q=q, part="id,snippet", maxResults=50, type="video",
                    order="relevance", publishedAfter=PUBLISHED_AFTER,
                    pageToken=page_token
                ).execute()
                sq_used += SEARCH_COST
                track_quota(SEARCH_COST, f"search '{q[:25]}'")
            except Exception as e:
                if check_quota_error(e):
                    print("  QUOTA ESAURITA.")
                else:
                    print(f"  {e}")
                break

            for item in resp.get("items", []):
                v_id = item["id"]["videoId"]
                if v_id in seen:
                    continue
                title = item["snippet"]["title"]
                if not is_relevant(title, item["snippet"].get("description", "")):
                    continue
                seen.add(v_id)
                video_map[v_id] = {
                    "Titolo_Video": title,
                    "Canale":       item["snippet"]["channelTitle"],
                    "Link":         f"https://www.youtube.com/watch?v={v_id}",
                    "Data_Video":   item["snippet"]["publishedAt"],
                    "Is_Short":     None,
                    "Topic":        "GOVERNO",
                }

            page_count += 1
            page_token = resp.get("nextPageToken")
            print(f"  p{page_count}: +{len(video_map)} totali")
            if not page_token:
                break
            time.sleep(0.3)

    return video_map


def enrich_with_duration(video_map):
    """Aggiunge Is_Short. Si ferma se la quota si esaurisce a metà."""
    global quota_exhausted
    ids = [v for v, m in video_map.items() if m.get("Is_Short") is None]
    if not ids or quota_exhausted:
        return video_map

    print(f"Durata per {len(ids)} video...")
    for i in range(0, len(ids), 50):
        batch = ids[i:i+50]
        try:
            resp = youtube.videos().list(part="contentDetails", id=",".join(batch)).execute()
            track_quota(VIDEO_COST, "duration")
            for item in resp.get("items", []):
                dur = item["contentDetails"]["duration"]
                video_map[item["id"]]["Is_Short"]   = is_short(dur)
                video_map[item["id"]]["Durata_ISO"] = dur
        except Exception as e:
            if check_quota_error(e):
                break
            print(f"  {e}")
        time.sleep(0.2)
    return video_map


def extract_comments(video_map, known_ids, last_dates, known_keys):
    """Estrazione commenti con monitoraggio quota; si ferma se restano < 50 unità."""
    global quota_exhausted
    new_rows = []
    for idx, (v_id, meta) in enumerate(video_map.items(), 1):
        if quota_exhausted or quota_remaining() < 50:
            print(f"  Stop: quota insufficiente. Raccolti {len(new_rows):,}")
            break
        is_known  = v_id in known_ids
        last_date = last_dates.get(v_id)
        print(f"[{idx}/{len(video_map)}] {'D' if is_known else '+'} {meta['Titolo_Video'][:45]}...")
        try:
            page_token, page_count = None, 0
            while page_count < MAX_COMMENT_PAGES:
                if quota_exhausted:
                    break
                kwargs = dict(part="snippet,replies", videoId=v_id, maxResults=100,
                              textFormat="plainText", order="time", pageToken=page_token)
                if is_known and last_date:
                    kwargs["publishedAfter"] = last_date

                resp = youtube.commentThreads().list(**kwargs).execute()
                track_quota(COMMENT_COST, "comments")
                page_new = 0

                for item in resp["items"]:
                    top  = item["snippet"]["topLevelComment"]["snippet"]
                    ckey = (v_id, top["authorDisplayName"],
                            str(pd.to_datetime(top["publishedAt"], utc=True)))
                    if ckey not in known_keys:
                        known_keys.add(ckey)
                        new_rows.append({
                            **meta, "Video_ID": v_id,
                            "Tipo_Commento": "Principale", "Padre_ID": None,
                            "Autore":        top["authorDisplayName"],
                            "Testo":         top["textDisplay"],
                            "Likes":         top["likeCount"],
                            "Data_Commento": top["publishedAt"],
                        })
                        page_new += 1

                    if "replies" in item:
                        for reply in item["replies"]["comments"]:
                            r    = reply["snippet"]
                            rkey = (v_id, r["authorDisplayName"],
                                    str(pd.to_datetime(r["publishedAt"], utc=True)))
                            if rkey not in known_keys:
                                known_keys.add(rkey)
                                new_rows.append({
                                    **meta, "Video_ID": v_id,
                                    "Tipo_Commento": "Risposta",
                                    "Padre_ID":      item["id"],
                                    "Autore":        r["authorDisplayName"],
                                    "Testo":         r["textDisplay"],
                                    "Likes":         r["likeCount"],
                                    "Data_Commento": r["publishedAt"],
                                })
                                page_new += 1

                if is_known and page_new == 0:
                    break   # nessuna novità per video noto → stop anticipato
                page_count += 1
                page_token  = resp.get("nextPageToken")
                if not page_token:
                    break
                time.sleep(0.2)

        except Exception as e:
            print(f"  {e}")

    print(f"\nNuovi commenti: {len(new_rows):,}")
    return pd.DataFrame(new_rows)

### §3.2 — Esecuzione pipeline governo

In [ ]:
df_existing, known_ids, last_dates, known_keys = load_existing_state(OUTPUT_FILE)
video_map = collect_video_ids(QUERIES, known_ids)
video_map = enrich_with_duration(video_map)

df_new = extract_comments(video_map, known_ids, last_dates, known_keys)

if df_new.empty:
    print("\nNessun nuovo commento.")
else:
    df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    before = len(df_combined)
    df_combined.drop_duplicates(
        subset=["Video_ID", "Autore", "Data_Commento", "Testo"], inplace=True
    )
    if before != len(df_combined):
        print(f"  Rimossi {before - len(df_combined)} duplicati.")
    df_combined.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(f"\nSalvato {OUTPUT_FILE}: {len(df_combined):,} righe ({len(df_new):,} nuove)")

## §4 — Dataset Stampa

Il dataset stampa (`dataset_articoli_referendum.csv`) è stato raccolto **fuori da Kaggle**: scraping mirato sui siti dei principali quotidiani italiani per il periodo ottobre 2025 – marzo 2026, focus su articoli che menzionano il referendum sulla separazione delle carriere.

**Schema**:
| Colonna | Descrizione |
|---------|-------------|
| `Data_Articolo` | Data di pubblicazione |
| `Testata` | Nome del giornale (Il Fatto Quotidiano, Il Giornale, Libero, ...) |
| `Titolo` | Titolo dell'articolo |
| `STANCE` | Etichetta annotata: POS (pro-riforma) / NEG (contro-riforma) / NEU (informativo) |

**Volume**: 3.424 articoli da 30+ testate.

**Annotazione**: stessa scala STANCE del YouTube (POS/NEG/NEU), così che i tre dataset siano confrontabili nelle analisi di Granger e turning points.

La cella sotto carica e ispeziona il dataset per verificare integrità.

In [ ]:
import pandas as pd

PATH_STAMPA = "/kaggle/input/datasets/francescochiarolanza/titoli-giornali-referendum-giustizia/dataset_articoli_referendum.csv"

# Auto-detect del separatore (alcune versioni usano \t o ;)
df_st = pd.read_csv(PATH_STAMPA, dtype=str, encoding="utf-8-sig",
                    engine="python", on_bad_lines="skip")
if len(df_st.columns) <= 3:
    df_st = pd.read_csv(PATH_STAMPA, sep=";", dtype=str, encoding="utf-8-sig",
                        engine="python", on_bad_lines="skip")

df_st = df_st.drop(columns=[c for c in df_st.columns if "Unnamed" in c], errors="ignore")
print(f"Articoli totali: {len(df_st):,}")
print(f"Colonne: {df_st.columns.tolist()}")

# Distribuzione stance
if "STANCE" in df_st.columns:
    df_st["STANCE"] = df_st["STANCE"].astype(str).str.strip().str.upper()
    print("\nDistribuzione STANCE:")
    print(df_st["STANCE"].value_counts())

# Distribuzione testate (top 10)
testata_col = next((c for c in df_st.columns if "testata" in c.lower()), None)
if testata_col:
    print(f"\nTop 10 testate per volume:")
    print(df_st[testata_col].value_counts().head(10))

## §5 — Associazione Padre-Figlio

L'API YouTube restituisce commenti principali e risposte come record separati, con un `Padre_ID` che lega la risposta al commento principale. Per il modello BERT è importante fornire il **testo del commento padre come contesto** della risposta: i thread di risposta usano spesso pronomi impliciti (`anche loro`, `come dici tu`, `ma se è così`) che senza il padre sono ambigui.

**Approccio**: scorri i commenti **in ordine** per ogni video. Tieni traccia dell'ultimo principale visto. Quando incontri una risposta, assegnale il testo del principale corrente. È un'approssimazione **posizionale** (presuppone che le risposte seguano subito il padre nell'ordine restituito dall'API) — copre l'80%+ dei thread.

**Output**: nuovo CSV `referendum_con_padre.csv` con colonna aggiuntiva `Testo_Padre`.

In [ ]:
import pandas as pd
import numpy as np

# Carica il dataset YouTube referendum (output di §2)
df = pd.read_csv(
    "/kaggle/input/datasets/francescochiarolanza/commenti-youtube-referendum-giustizia/referendum_giustizia_full_debate.csv",
    encoding="utf-8-sig", dtype=str, low_memory=False
)
df = df.drop(columns=[c for c in df.columns if "Unnamed" in c], errors="ignore")
print(f"Colonne: {df.columns.tolist()}")
print(f"Righe:   {len(df):,}")

df["Testo_Padre"] = np.nan
matched   = 0
unmatched = 0

# Itera per video: dentro ogni gruppo, scorri in ordine e abbina padre-risposta
for video_id, group in df.groupby("Video_ID"):
    idx = group.index.tolist()
    current_padre = None   # testo dell'ultimo commento principale visto

    for i in idx:
        row = df.loc[i]
        if row["Tipo_Commento"] == "Principale":
            current_padre = row["Testo"]
        elif row["Tipo_Commento"] == "Risposta":
            # Abbinamento posizionale: ultimo principale visto = padre presunto
            if current_padre is not None:
                df.at[i, "Testo_Padre"] = current_padre
                matched += 1
            else:
                unmatched += 1   # risposta senza padre identificabile (rara)

print(f"\nRisposte con padre trovato: {matched:,}")
print(f"Risposte senza padre:        {unmatched:,}")
print(f"Commenti principali:         {(df['Tipo_Commento']=='Principale').sum():,}")
print(f"Totale righe:                {len(df):,}")

# Verifica qualitativa su un campione
print("\n--- ESEMPIO: risposta + padre abbinato ---")
for _, row in df[df["Testo_Padre"].notna()].head(3).iterrows():
    print(f"  PADRE:    {str(row['Testo_Padre'])[:100]}")
    print(f"  RISPOSTA: {str(row['Testo'])[:100]}")
    print()

# Salva con sep=";" per coerenza con la pipeline successiva (training BERT)
df.to_csv("/kaggle/working/referendum_con_padre.csv",
          index=False, encoding="utf-8-sig", sep=";")
print("Salvato: /kaggle/working/referendum_con_padre.csv")

---

## Output di questo notebook

Tre CSV pronti per il notebook 2 (training):

| File | Contenuto | Usato in |
|------|-----------|----------|
| `Dataset Tesi Aggiornato.csv` | YouTube referendum (con `Is_Short`, `Padre_ID`) | §5 di questo notebook |
| `governo_meloni_consenso.csv` | YouTube governo (con `Is_Short`) | Notebook 2 — training governo |
| `referendum_con_padre.csv` | YouTube referendum + colonna `Testo_Padre` | Notebook 2 — training referendum |

Il dataset stampa è già pronto come input Kaggle (raccolto esternamente).

→ Procedi al **Notebook 2 — Allenamento Modelli**.